In [ ]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import numpy as np
import astropy.units as u
import bd_support as sup

from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

# To see what clouds are availible
# vj.available()

In [ ]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
opaci_path  = None # Opacity db
# opcai_path  = '/groups/tkaralidi/opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\cloudy_spectra_code\outputs")
# output_path = 'home/al864695/ouputs'

# Things that will remain constant
clouds      = ['Al2O3', 'CaTiO3', 'CH4', 'Cr', 'Fe',
               'H2O', 'KCl', 'Mg2SiO4', 'MgSiO3', 'MnS',
               'NH3', 'Na2S', 'TiO2', 'ZnS']

# Constant values
wav_range   = [0.3, 5.0] # microns
MH          = 1.0        # [M/H] metallicity factor ~ solar
MU          = 2.36       # Average MU
R           = 300        # resolution
# R           = 5000

In [ ]:
##### METHOD BROWN DWARF SPECTRUM

def bd_spectrum(Teff, logg, fsed, kzz):
    """
    Compute a BD emission spectrum with Virga clouds.
    """

    # Opacity & inputs
    opa    = jdi.opannection(wav_range, opaci_path)
    bd     = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)
    # Convert log g [cgs] to grav [m s^-2]
    gravity = 10**logg * 1e-2   # 1 cm s^-2 = 0.01 m s^-2
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # Inject corrected TP
    sup.inject_corr(bd, Teff, logg, fsed)

    # Inject Kzz (match pressure grid length)
    prof   = bd.inputs['atmosphere']['profile']
    P      = np.asarray(prof["pressure"], float)
    T      = np.asarray(prof['temperature'], float)
    bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

    # Recommended gas
    rec    = vj.recommend_gas(P, T, MH, MU, plot=False)
    
    # Remove THESE species because not supported
    exclude = {'CaAl12O19', 'SiO2'}
    clouds = [sp for sp in rec if sp not in exclude]
    
    # Clouds
    bd.virga(clouds, virga_path, fsed, mh=MH, mmw=MU)
    out    = bd.spectrum(opa, full_output=True)

    wn, fl = out["wavenumber"], out["thermal"]  # cm^-1 and erg cm^-2 s^-1 cm^-1
    
    # Regrid in wavenumber space because F is defined per wavenumber
    wn, fl = jdi.mean_regrid(wn, fl, R=R)

    # Convert wavenumber [cm^-1] to wavelength [micron]
    w_um = 1e4 / wn

    return w_um, fl

In [ ]:
##### GENERATE AND SAVE SPECTRUM (MARGE format, single-case files)

# Define ranges to match your grid
Teff_min, Teff_max = 1300, 1800
logg_min, logg_max = 3.0, 5.0
fsed_min, fsed_max = 2.0, 4.0
kzz_min,  kzz_max  = 1e9, 1e10

N_samples = 50  # how many random points to draw

Teff_s = np.random.uniform(Teff_min, Teff_max, N_samples)
logg_s = np.random.uniform(logg_min, logg_max, N_samples)
fsed_s = np.random.uniform(fsed_min, fsed_max, N_samples)
kzz_s  = np.random.uniform(kzz_min , kzz_max , N_samples)

for Teff_i, logg_i, fsed_i, kzz_i in zip(Teff_s, logg_s, fsed_s, kzz_s):

    # Run spectrum
    W_i, F_i = bd_spectrum(Teff_i, logg_i, fsed_i, kzz_i)

    # Filename encodes the parameters; ML will parse from name
    fname  = f"t{float(Teff_i)}g{float(logg_i)}f{float(fsed_i)}{sup.format_kzz(kzz_i)}.npz"
    fpath  = output_path / fname

    # MARGE wants 'x' and 'y' keys. Since x and y have the same first-dimension length,
    # add a leading axis of size 1 to mark this as a *single* case.
    # x = wavelength [micron], y = flux per wavenumber [erg cm^-2 s^-1 (cm^-1)^-1]
    x_single = np.asarray(W_i, dtype=np.float32)[None, ...]  # shape (1, N)
    y_single = np.asarray(F_i, dtype=np.float32)[None, ...]  # shape (1, N)

    np.savez_compressed(fpath, x=x_single, y=y_single)

print(f"Saved {N_samples} random files to {output_path}")
print(f"Saved {len(Teff_s)*len(logg_s)*len(fsed_s)*len(kzz_s)}")